# Lab 4: Clustering

In this lab we implement k-means, agglomerative clustering and the k-NN distance curve of DBSCAN from scratch with NumPy, and compare them with DBSCAN and HDBSCAN. A bonus uses k-means on a photo taken on Mars.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial.distance import cdist
from scipy.cluster import hierarchy
from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.metrics import adjusted_rand_score
plt.style.use("seaborn-v0_8")

## Helpers

`cdist(A, B)` (from SciPy) returns the table of distances between every row of `A` and every row of `B`, with shape `(len(A), len(B))`. We use it in every exercise.

**`plot_clusters`** draws the points coloured by their cluster. Points labelled −1 are noise (DBSCAN and HDBSCAN) and are drawn in grey. Run the cell and move on.

In [ ]:
def plot_clusters(X, labels, title=None, ax=None):
    """Scatter the 2D points X, coloured by their cluster label; label -1 (noise) is drawn in grey.

    ax: optional matplotlib axis, to draw inside a grid of subplots
    """
    # create a new figure if no axis is given
    if ax is None:
        ax = plt.figure(figsize=(6, 5)).gca()

    # noise points in grey, clustered points coloured by their label
    noise = labels == -1
    ax.scatter(X[noise, 0], X[noise, 1], color="0.7", s=10)
    ax.scatter(X[~noise, 0], X[~noise, 1], c=labels[~noise], cmap="tab10", s=20)

    # title with the number of clusters
    n_clusters = len(np.unique(labels[~noise]))
    ax.set_title(f"{title}: {n_clusters} clusters" if title else f"{n_clusters} clusters")

## Clustering two datasets

We use two datasets without labels:
- **toy data**: two half-moons and two round blobs (100 points);
- **noisy data**: 2309 points with clusters of different shapes and densities, plus background noise.

In [ ]:
moons, _ = make_moons(n_samples=50, noise=0.05, random_state=0)
blobs, _ = make_blobs(n_samples=50, centers=[(-0.75, 2.25), (1.0, 2.0)], cluster_std=0.25, random_state=0)
X_toy = np.vstack([moons, blobs])
X_noisy = np.load("clusterable_data.npy")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_toy[:, 0], X_toy[:, 1], s=20)
axes[0].set_title("Toy data")
axes[1].scatter(X_noisy[:, 0], X_noisy[:, 1], s=10)
axes[1].set_title("Noisy data")
plt.show()

### 1.1 K-means

K-means looks for $K$ centres $\mu_1, \dots, \mu_K$ and puts each point in the cluster of its nearest centre. It minimises the **inertia**, the sum of the squared distances between each point and its centre:
$$
\text{inertia} = \sum_{i=1}^{n} \| x_i - \mu_{c_i} \|^2 ,
$$
where $c_i$ is the cluster of point $i$. **Lloyd's algorithm** starts from $K$ random points of the data as centres, and repeats two steps until the centres stop moving:
1. **assign**: each point goes to the cluster of its nearest centre;
2. **update**: each centre moves to the mean of the points of its cluster.

**Exercise 1.** Complete `KMeans`.

In [ ]:
class KMeans:
    """K-means clustering with Lloyd's algorithm."""

    def __init__(self, n_clusters, n_iter=100, seed=0):
        self.n_clusters = n_clusters  # K
        self.n_iter = n_iter          # maximum number of iterations
        self.seed = seed              # seed of the random starting centres
        self.centers_ = None          # (K, num_features) array of the centres
        self.labels_ = None           # cluster of every training point
        self.inertia_ = None          # sum of the squared distances to the centres

    def predict(self, X):
        """Index of the nearest centre for every row of X."""
        # TODO
        # Step 1: distances between every point and every centre: cdist(X, self.centers_), shape (num_samples, K)
        # Step 2: return the index of the smallest distance of each row (np.argmin with axis=1)

    def fit(self, X):
        """Run Lloyd's algorithm on the data X, of shape (num_samples, num_features)."""
        # TODO
        # Step 1: starting centres: K different random rows of X
        #         rng = np.random.RandomState(self.seed)
        #         rows = rng.choice(len(X), self.n_clusters, replace=False), then self.centers_ = X[rows]
        # Step 2: repeat at most self.n_iter times:
        #     a) assign: labels = self.predict(X)
        #     b) update: new centres, the mean of the points of each cluster k: X[labels == k].mean(axis=0)
        #     c) if the new centres are equal to the old ones (np.allclose), stop the loop (break)
        #     d) otherwise, store the new centres in self.centers_
        # Step 3: final labels, self.labels_ = self.predict(X)
        # Step 4: inertia, the sum of the squared distances between each point and its centre:
        #         self.inertia_ = np.sum((X - self.centers_[self.labels_]) ** 2)
        return self

Run k-means with $K = 4$ on the toy data and $K = 6$ on the noisy data:

In [ ]:
kmeans_toy = KMeans(n_clusters=4)
kmeans_toy.fit(X_toy)
kmeans_noisy = KMeans(n_clusters=6)
kmeans_noisy.fit(X_noisy)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_clusters(X_toy, kmeans_toy.labels_, title="k-means, toy data", ax=axes[0])
plot_clusters(X_noisy, kmeans_noisy.labels_, title="k-means, noisy data", ax=axes[1])
plt.show()

# the result depends on the random starting centres
for seed in [0, 1, 2]:
    model = KMeans(n_clusters=6, seed=seed)
    model.fit(X_noisy)
    print(f"Noisy data, seed {seed}: inertia = {model.inertia_:.2f}")

K-means cuts the moons in the wrong places: it looks for round clusters around centres. On the noisy data, every point, including the noise, is forced into a cluster, and different starting centres give different results, with different inertias.

### 1.2 Agglomerative clustering

Agglomerative clustering starts with every point in its own cluster, and merges the two closest clusters, one merge at a time. After each merge, the distances between the new cluster and the others are updated. This update defines the **linkage**:
- **single** linkage: distance between the closest points of the two clusters, $d(i \cup j,\, k) = \min\big(d(i, k),\, d(j, k)\big)$;
- **complete** linkage: distance between the farthest points, $d(i \cup j,\, k) = \max\big(d(i, k),\, d(j, k)\big)$.

Stopping when $K$ clusters remain is the same as cutting the dendrogram at the height where $K$ branches remain.

**Exercise 2.** Complete `agglomerative`.

In [ ]:
def agglomerative(X, n_clusters, linkage="single"):
    """Agglomerative clustering of X, stopped when n_clusters clusters remain.

    Parameters:
    -----------
    X: (num_samples, num_features) np.array
    n_clusters: int
    linkage: "single" or "complete"

    Returns:
    --------
    labels: (num_samples,) np.array of cluster labels 0, 1, ..., n_clusters - 1
    """
    # TODO
    # Step 1: distance matrix D = cdist(X, X), with np.inf on the diagonal (np.fill_diagonal),
    #         so that a cluster is never merged with itself
    # Step 2: labels = np.arange(len(X)): every point is its own cluster, named by its index
    # Step 3: repeat len(X) - n_clusters times (one merge each time):
    #     a) closest pair of clusters: i, j = np.unravel_index(np.argmin(D), D.shape)
    #     b) distances of the merged cluster to the others:
    #        np.minimum(D[i], D[j]) for single linkage, np.maximum(D[i], D[j]) for complete linkage
    #     c) the merged cluster keeps the name i: put these distances in row i and column i, then D[i, i] = np.inf
    #     d) cluster j no longer exists: row j and column j become np.inf
    #     e) the points of cluster j now belong to cluster i: labels[labels == j] = i
    # Step 4: renumber the labels 0, 1, ..., n_clusters - 1: return np.unique(labels, return_inverse=True)[1]

Compare single and complete linkage on the toy data, with $K = 4$. As a check, the adjusted Rand index between our clusters and those of scikit-learn's `AgglomerativeClustering` should be 1 (the same clusters, whatever their names):

In [ ]:
single = agglomerative(X_toy, 4, linkage="single")
complete = agglomerative(X_toy, 4, linkage="complete")

for name, ours in [("single", single), ("complete", complete)]:
    sklearn_labels = AgglomerativeClustering(n_clusters=4, linkage=name).fit_predict(X_toy)
    print(f"{name:8s} linkage, same clusters as scikit-learn: {adjusted_rand_score(sklearn_labels, ours):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_clusters(X_toy, single, title="single linkage", ax=axes[0])
plot_clusters(X_toy, complete, title="complete linkage", ax=axes[1])
plt.show()

Single linkage separates the two moons: it follows chains of close points. Complete linkage prefers compact, round clusters, like k-means, and cuts the moons.
The lecture's "Four linkages" slide shows the other side: with noise between two clusters, one bridge of points is enough for single linkage to join them, while complete linkage resists it.

The full history of the merges is the **dendrogram**. Cutting it where 4 branches remain gives the 4 single-linkage clusters:

In [ ]:
plt.figure(figsize=(14, 4))
hierarchy.dendrogram(hierarchy.linkage(X_toy, method="single"), no_labels=True)
plt.title("Single-linkage dendrogram of the toy data")
plt.ylabel("merge distance")
plt.show()

### 1.3 DBSCAN

DBSCAN finds clusters as dense regions. With two parameters, a radius $\varepsilon$ and a number of points MinPts:
- a **core** point has at least MinPts points within distance $\varepsilon$;
- a **border** point is within $\varepsilon$ of a core point, without being a core point itself;
- the other points are **noise**: they are not put in any cluster.

Clusters are groups of connected core points, with their border points.

To choose $\varepsilon$, we use the **k-NN distance curve**: for every point, the distance to its $k$-th nearest neighbour (with $k$ = MinPts), sorted from smallest to largest. Points in dense regions have small values, noise points have large values, and a good $\varepsilon$ is where the curve starts to rise sharply.

**Exercise 3.** Complete `knn_distances`.

In [ ]:
def knn_distances(X, k):
    """Distance from every point to its k-th nearest neighbour, sorted from smallest to largest.

    Parameters:
    -----------
    X: (num_samples, num_features) np.array
    k: int

    Returns:
    --------
    distances: (num_samples,) np.array
    """
    # TODO
    # Step 1: distance matrix D = cdist(X, X)
    # Step 2: sort each row (np.sort with axis=1): column 0 is the point itself (distance 0),
    #         and column k is the distance to its k-th nearest neighbour
    # Step 3: take column k, and return it sorted (np.sort)

The curve for the noisy data, with MinPts = 5:

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(knn_distances(X_noisy, k=5))
plt.ylim(0, 0.1)
plt.xlabel("points, sorted")
plt.ylabel("distance to the 5th nearest neighbour")
plt.show()

The curve starts to rise sharply around 0.03. The effect of $\varepsilon$ on DBSCAN (scikit-learn), with MinPts = 5:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
for ax, eps in zip(axes, [0.015, 0.03, 0.06]):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_noisy)
    plot_clusters(X_noisy, labels, title=f"DBSCAN, eps = {eps}, noise = {np.mean(labels == -1):.0%}", ax=ax)
plt.show()

No value of $\varepsilon$ works well here. With $\varepsilon = 0.015$ the dense parts are found, but 30% of the points are noise and the sparser clusters break into many small pieces. With $\varepsilon = 0.03$ and $0.06$ the clusters touch each other through the noise and merge into one big cluster. The clusters have different densities, so a single radius cannot fit all of them: the k-NN curve gives a starting value, not a guarantee.

### 1.4 HDBSCAN

HDBSCAN runs DBSCAN at every scale at once, builds the hierarchy of the clusters, and keeps the most **stable** ones. There is no $\varepsilon$ to choose, only the smallest cluster size we accept. Compare it with DBSCAN at $\varepsilon = 0.03$:

In [ ]:
dbscan_labels = DBSCAN(eps=0.03, min_samples=5).fit_predict(X_noisy)
hdbscan_labels = HDBSCAN(min_cluster_size=15).fit_predict(X_noisy)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_clusters(X_noisy, dbscan_labels, title="DBSCAN, eps = 0.03", ax=axes[0])
plot_clusters(X_noisy, hdbscan_labels, title="HDBSCAN, min_cluster_size = 15", ax=axes[1])
plt.show()

HDBSCAN finds the six clusters, each at its own density, and leaves the background as noise. DBSCAN, with one $\varepsilon$ for the whole dataset, either breaks them into pieces or merges them.

## Bonus (at home): something on Mars

This photo was taken on Mars by the Curiosity rover. Besides the ground and parts of the rover, there is something small on the ground. Can clustering find it?

In [ ]:
nasa = np.asarray(Image.open("im_nasa_reduced.jpg")) / 255  # (286, 320, 3) array, colours between 0 and 1

plt.figure(figsize=(8, 7))
plt.imshow(nasa)
plt.grid(False)
plt.show()

Instead of the three colours, we describe each pixel by two numbers that separate bright from dark, and grey from coloured:
$$
I = \frac{R + G + B}{3} \quad \text{(intensity)}, \qquad S = 1 - \frac{\min(R, G, B)}{I} \quad \text{(saturation)} .
$$

**Bonus exercise.** Complete `intensity_saturation`.

In [ ]:
def intensity_saturation(image):
    """Intensity and saturation of every pixel of an RGB image with colours between 0 and 1.

    Parameters:
    -----------
    image: (height, width, 3) np.array

    Returns:
    --------
    I, S: two (height, width) np.arrays
    """
    # TODO
    # Step 1: the three colour channels: R, G, B = image[..., 0], image[..., 1], image[..., 2]
    # Step 2: intensity I = (R + G + B) / 3
    # Step 3: saturation S = 1 - min(R, G, B) / I   (np.minimum takes the minimum of two arrays)
    # Step 4: return I, S

K-means with $K = 3$ on the pixels described by $(I, S)$. On the right, only the pixels of the brightest cluster are shown in white:

In [ ]:
I, S = intensity_saturation(nasa)
features = np.column_stack([I.ravel(), S.ravel()])  # one row (I, S) per pixel

kmeans_nasa = KMeans(n_clusters=3)
kmeans_nasa.fit(features)
segmentation = kmeans_nasa.labels_.reshape(I.shape)
brightest = np.argmax(kmeans_nasa.centers_[:, 0])  # cluster with the largest intensity

fig, axes = plt.subplots(1, 3, figsize=(27, 8))
axes[0].imshow(nasa)
axes[0].set_title("Photo")
axes[1].imshow(segmentation, cmap="tab10")
axes[1].set_title("k-means, K = 3")
axes[2].imshow(segmentation == brightest, cmap="gray")
axes[2].set_title("Brightest cluster")
for ax in axes:
    ax.grid(False)
plt.show()

**Question.** What does each cluster correspond to? Where is the small object on the ground?

**Answer:**